#Aaj hum manually C=1.0 choose nahi karenge. GridSearchCV automatically multiple parameter combinations test karega using cross-validation, aur best combination find karega.

Aaj ka experiment:
- Logistic Regression
- C: 0.01, 0.1, 1, 10, 100
- class_weight: None, balanced
- 5-fold cross-validation
- Malignant Recall ko scoring metric banayenge

In [2]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import recall_score, make_scorer

# Load dataset
data = load_breast_cancer()
X = data.data
y = data.target

# Convert labels: 0 = Malignant, 1 = Benign
y = 1 - y

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training Size:", X_train.shape)
print("Testing Size:", X_test.shape)

# Pipeline
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000))
])

# Parameter grid
param_grid = {
    "model__C": [0.01, 0.1, 1, 10, 100],
    "model__class_weight": [None, "balanced"]
}

# Malignant Recall scorer (class 0)
malignant_recall = make_scorer(recall_score, pos_label=0)

# Cross-validation
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Grid Search
grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring=malignant_recall,
    cv=cv,
    n_jobs=-1,
    verbose=1
)

# Fit
grid_search.fit(X_train, y_train)

# Results
print("\nBest Parameters:", grid_search.best_params_)
print("Best CV Malignant Recall:", grid_search.best_score_)
print("Best Model:", grid_search.best_estimator_)

Training Size: (455, 30)
Testing Size: (114, 30)
Fitting 5 folds for each of 10 candidates, totalling 50 fits

Best Parameters: {'model__C': 0.01, 'model__class_weight': None}
Best CV Malignant Recall: 0.9964912280701753
Best Model: Pipeline(steps=[('scaler', StandardScaler()),
                ('model', LogisticRegression(C=0.01, max_iter=1000))])


#Cell 2 — Evaluate the Best Model on Test Data

In [4]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# Best model predictions are already available
y_pred_best = best_model.predict(X_test)

# Correct positive class: Malignant = 1
print("Correct Malignant Evaluation")
print("=" * 40)

print("Accuracy:", accuracy_score(y_test, y_pred_best))
print("Malignant Precision:", precision_score(y_test, y_pred_best, pos_label=1))
print("Malignant Recall:", recall_score(y_test, y_pred_best, pos_label=1))
print("Malignant F1:", f1_score(y_test, y_pred_best, pos_label=1))

print("\nConfusion Matrix (Benign=0, Malignant=1):")
print(confusion_matrix(y_test, y_pred_best))

Correct Malignant Evaluation
Accuracy: 0.9473684210526315
Malignant Precision: 1.0
Malignant Recall: 0.8571428571428571
Malignant F1: 0.9230769230769231

Confusion Matrix (Benign=0, Malignant=1):
[[72  0]
 [ 6 36]]


In [5]:
from sklearn.metrics import make_scorer, recall_score
from sklearn.model_selection import GridSearchCV

# Correct scorer: Malignant = class 1
malignant_recall_scorer = make_scorer(
    recall_score,
    pos_label=1
)

# Grid Search
grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring=malignant_recall_scorer,
    cv=cv,
    n_jobs=-1,
    verbose=1
)

# Train
grid_search.fit(X_train, y_train)

print("\nCorrect GridSearch Results")
print("Best Parameters:", grid_search.best_params_)
print("Best CV Malignant Recall:", grid_search.best_score_)

Fitting 5 folds for each of 10 candidates, totalling 50 fits

Correct GridSearch Results
Best Parameters: {'model__C': 1, 'model__class_weight': 'balanced'}
Best CV Malignant Recall: 0.9588235294117646


#Final Test Evaluation

In [6]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# Best model
best_model = grid_search.best_estimator_

# Predict on test set
y_pred_best = best_model.predict(X_test)

# Evaluation
print("Correct GridSearchCV — Test Results")
print("=" * 45)

print("Accuracy:", accuracy_score(y_test, y_pred_best))
print("Malignant Precision:", precision_score(y_test, y_pred_best, pos_label=1))
print("Malignant Recall:", recall_score(y_test, y_pred_best, pos_label=1))
print("Malignant F1:", f1_score(y_test, y_pred_best, pos_label=1))

print("\nConfusion Matrix (Benign=0, Malignant=1):")
print(confusion_matrix(y_test, y_pred_best, labels=[0, 1]))

Correct GridSearchCV — Test Results
Accuracy: 0.9736842105263158
Malignant Precision: 0.975609756097561
Malignant Recall: 0.9523809523809523
Malignant F1: 0.963855421686747

Confusion Matrix (Benign=0, Malignant=1):
[[71  1]
 [ 2 40]]
